<a href="https://colab.research.google.com/github/juanpajaro/intro_ciencia_datos_quimica/blob/main/Ejemplo_clasificacion_medicamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Contexto del Proyecto**

Como principiante en el mundo de la ciencia de datos, este proyecto representa una excelente oportunidad para implementar diversas técnicas predictivas. El objetivo principal es determinar con precisión qué fármaco es el más adecuado para un paciente, basándose en su perfil clínico y biológico.

---
### **Objetivos Clave:**
* **Exploración de Datos:** Entender la relación entre los indicadores químicos (como el ratio sodio-potasio) y la eficacia del medicamento.
* **Modelado:** Aplicar algoritmos de clasificación para predecir la variable objetivo `Drug`.
* **Precisión:** Evaluar qué técnica ofrece los resultados más confiables para la seguridad del paciente.

In [ ]:
#Numpy es una libreria para el manejo de arreglos numericos
import numpy as np
#Pandas es una libreria para el manejo de datos tabulares
import pandas as pd
#Matplotlib es una libreria para graficar los datos
import matplotlib.pyplot as plt
#Seaborn es otra libreria para graficar los datos
import seaborn as sns
#Os es una liberia para interacturar con el sistema operativo
import os

# **Cargue de datos**

In [ ]:
#Carga los datos
df_drug = pd.read_csv('drug200.csv')
df_drug.head()

### **Diccionario de Datos: Clasificación de Medicamentos**

La siguiente tabla describe las variables del conjunto de datos utilizado para la predicción de tipos de fármacos basados en perfiles de pacientes.

| Variable | Descripción | Datos de Ejemplo |
| :--- | :--- | :--- |
| **Age** | Edad del paciente | 23; 47; ... |
| **Sex** | Género del paciente (F: Femenino, M: Masculino) | F; M; ... |
| **BP** | Niveles de presión sanguínea (HIGH, NORMAL, LOW) | HIGH; NORMAL; LOW; ... |
| **Cholesterol** | Niveles de colesterol (HIGH, NORMAL) | 1.4; 1.3; ... |
| **Na_to_K** | Relación de sodio a potasio en la sangre | 25.355; 13.093; ... |
| **Drug** | Tipo de fármaco prescrito (Variable objetivo) | DrugY; drugC; ... |

---

In [ ]:
#numero de columnas y filas
df_drug.info()
#¿Existe volumen y dimension en los datos?
#¿Es una pregunta compleja?

#**1. Gestion de datos**

## **Analisis exploratorio de datos**

###Distribución de las drogas

In [ ]:
sns.set_theme(style="darkgrid")
sns.countplot(y="Drug", data=df_drug, palette="flare")
plt.ylabel('Drug Type')
plt.xlabel('Total')
plt.show()

In [ ]:
df_drug['Drug'].value_counts()

###Distribución por sexo

In [ ]:
sns.set_theme(style="darkgrid")
sns.countplot(x="Sex", data=df_drug, palette="rocket")
plt.xlabel('Gender (F=Female, M=Male)')
plt.ylabel('Total')
plt.show()

In [ ]:
df_drug['Sex'].value_counts()

###Distribución de la presión sanguinea

In [ ]:
sns.set_theme(style="darkgrid")
sns.countplot(y="BP", data=df_drug, palette="crest")
plt.ylabel('Blood Pressure')
plt.xlabel('Total')
plt.show()

In [ ]:
df_drug['BP'].value_counts()

###Distribución de los niveles de colesterol

In [ ]:
#crear instruccion

###Distribución por drogas según sexo

In [ ]:
#crar instruccion

###Distribución de la presion sanguinea según los niveles de colesterol

In [ ]:
#crear instruccion

###Relación sodia a potación según edad diferenciada por sexo

In [ ]:
#crear instruccion

##**Preparacion de los datos**

In [ ]:
#Agrupamos la edad
bin_age = [0, 19, 29, 39, 49, 59, 69, 80]
category_age = ['<20s', '20s', '30s', '40s', '50s', '60s', '>60s']
df_drug['Age_binned'] = pd.cut(df_drug['Age'], bins=bin_age, labels=category_age)
df_drug = df_drug.drop(['Age'], axis = 1)

In [ ]:
#Agrupamos la relación sodio a potacion
bin_NatoK = [0, 9, 19, 29, 50]
category_NatoK = ['<10', '10-20', '20-30', '>30']
df_drug['Na_to_K_binned'] = pd.cut(df_drug['Na_to_K'], bins=bin_NatoK, labels=category_NatoK)
df_drug = df_drug.drop(['Na_to_K'], axis = 1)

####¿Por qué hacemos la agrupación anterior?

###División del conjunto de datos: entrenamiento y pruba

In [ ]:
from sklearn.model_selection import train_test_split

X = df_drug.drop(["Drug"], axis=1)
y = df_drug["Drug"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 0)

## **Ingeniería de Características (Feature Engineering)**

En esta etapa, aplicamos la técnica de **One-Hot Encoding**. Este método consiste en transformar las variables categóricas (como el género o los niveles de presión arterial) en un formato numérico binario.

¿Por qué lo hacemos? La mayoría de los algoritmos de Machine Learning no pueden interpretar texto directamente. Al convertir las categorías en columnas de ceros y unos ($0$ y $1$), permitimos que el modelo procese la información de manera matemática, mejorando significativamente la precisión de las predicciones.

In [ ]:
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

# Convertir los valores booleanos a enteros (0 or 1)
X_train = X_train.astype(int)
X_test = X_test.astype(int)

X_train.head()

## **Técnica SMOTE (Synthetic Minority Over-sampling Technique)**

Dado que la cantidad de registros para **'DrugY'** es significativamente mayor que la de otros tipos de fármacos, el conjunto de datos presenta un desequilibrio de clases. Para solucionar esto, aplicamos oversampling (sobremuestreo).

¿Por qué usamos SMOTE? Si entrenamos el modelo con datos desequilibrados, este tenderá a "especializarse" en la clase mayoritaria (DrugY) e ignorar las demás, lo que causa sobreajuste. SMOTE crea ejemplos sintéticos de las clases minoritarias en lugar de simplemente duplicar filas, permitiendo que el algoritmo aprenda mejor los patrones de los medicamentos menos frecuentes.


In [ ]:
from imblearn.over_sampling import SMOTE

X_train, y_train = SMOTE().fit_resample(X_train, y_train)

In [ ]:
sns.set_theme(style="darkgrid")
sns.countplot(y=y_train, data=df_drug, palette="mako_r")
plt.ylabel('Drug Type')
plt.xlabel('Total')
plt.show()

#**2. Entrenamiento de los algoritmos**

##**Arbol de decision**

In [ ]:
from sklearn.tree import DecisionTreeClassifier
DTclassifier = DecisionTreeClassifier(max_leaf_nodes=20, random_state=0)
DTclassifier.fit(X_train, y_train)

y_pred_dt = DTclassifier.predict(X_test)

##**Regresion logistica**

In [ ]:
from sklearn.linear_model import LogisticRegression
LRclassifier = LogisticRegression(solver='liblinear', max_iter=5000, random_state=0)
LRclassifier.fit(X_train, y_train)

y_pred_rl = LRclassifier.predict(X_test)


#**3. Evaluación de los modelos**

In [ ]:
from sklearn.metrics import accuracy_score
DTAcc = accuracy_score(y_pred_dt,y_test)
print('Exactiud del arbol de decisión: {:.2f}%'.format(DTAcc*100))

LRAcc = accuracy_score(y_pred_rl,y_test)
print('Exactitud de la regresion logistica: {:.2f}%'.format(LRAcc*100))